# Gaze-Cursor Temporal Lag in AdSERP

Huang, White & Buscher (CHI 2012) found cursor lags gaze by ~700ms (range 250-1000ms across individuals) on Bing SERPs. We have both streams in AdSERP — can we replicate and extend?

**Questions:**
1. What is the per-participant gaze-cursor lag in AdSERP?
2. Does lag correlate with TTI-to-first-scroll (our processing speed calibrator, r=0.77)?
3. Does lag predict satisfice/optimize behavior (regression rate)?
4. Is lag stable within participants across trials (trait vs state)?

**Method:** For each trial, cross-correlate fixation position (Y) with mouse position (Y) at varying temporal offsets. The offset minimizing RMSE is the lag. Average per participant.

**Data:** AdSERP fixation-data (page-space gaze, 150Hz fixation events) + mouse-movement-data (evtrack pageX/pageY, ~60Hz events). Both share Unix millisecond timestamps.

In [ ]:
import os, csv, math, json
import numpy as np
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt

DATA_DIR = Path('../AdSERP/data')
FIX_DIR = DATA_DIR / 'fixation-data'
MOUSE_DIR = DATA_DIR / 'mouse-movement-data'
META_DIR = DATA_DIR / 'trial-metadata'

assert FIX_DIR.exists(), f'Fixation data not found at {FIX_DIR}'
print(f'Trials: {len(list(FIX_DIR.glob("*.csv")))}')

In [ ]:
def load_fixations(trial_id):
    """Load fixation data: list of (t_ms, x, y, dur_ms)."""
    rows = []
    with open(FIX_DIR / f'{trial_id}.csv') as f:
        for row in csv.DictReader(f):
            t = int(float(row['timestamp']))
            x = float(row['FPOGX'])
            y = float(row['FPOGY'])
            d = float(row['FPOGD'])
            if math.isfinite(x) and math.isfinite(y) and d > 0:
                rows.append((t, x, y, d))
    return rows


def load_mouse_positions(trial_id):
    """Load mouse move/click events: list of (t_ms, x, y).
    Filters to positional events only (mousemove, mouseover, click, mousedown, mouseup)."""
    pos_events = ('mousemove', 'mouseover', 'mouseout', 'click', 'mousedown', 'mouseup')
    rows = []
    with open(MOUSE_DIR / f'{trial_id}.csv') as f:
        for row in csv.DictReader(f):
            evt = row['event'].strip()
            if evt not in pos_events:
                continue
            t = int(float(row['timestamp']))
            x = float(row['xpos'])
            y = float(row['ypos'])
            if math.isfinite(x) and math.isfinite(y):
                rows.append((t, x, y))
    return rows


def get_trial_ids():
    return sorted(p.stem for p in FIX_DIR.glob('*.csv'))


def parse_participant(trial_id):
    return trial_id.split('-')[0]


# Quick sanity check
tid = 'p004-b1-t1'
fix = load_fixations(tid)
mouse = load_mouse_positions(tid)
print(f'{tid}: {len(fix)} fixations, {len(mouse)} mouse events')
print(f'  Fix t range: {fix[0][0]} - {fix[-1][0]} ({(fix[-1][0]-fix[0][0])/1000:.1f}s)')
print(f'  Mouse t range: {mouse[0][0]} - {mouse[-1][0]}')

## Method: Cross-correlation lag estimation

For each trial:
1. Resample fixation positions onto a common 50ms timebase (center of each fixation held for its duration)
2. Resample mouse positions onto the same timebase (linear interpolation)
3. Compute RMSE between gaze Y and mouse Y at offsets from -2000ms to +2000ms (step 50ms)
4. The offset with minimum RMSE is the lag

We use Y coordinates because vertical position carries more information on a SERP (scroll direction). X varies less.

In [ ]:
def resample_fixations(fixations, t_start, t_end, step_ms=50):
    """Expand fixation events into a regular time series.
    Each fixation covers [t, t+dur). Returns (times, ys)."""
    times = np.arange(t_start, t_end, step_ms)
    ys = np.full(len(times), np.nan)
    for t, x, y, d in fixations:
        mask = (times >= t) & (times < t + d)
        ys[mask] = y
    return times, ys


def resample_mouse(mouse_events, times):
    """Interpolate mouse Y onto the given timebase."""
    if len(mouse_events) < 2:
        return np.full(len(times), np.nan)
    mt = np.array([m[0] for m in mouse_events], dtype=float)
    my = np.array([m[2] for m in mouse_events], dtype=float)  # Y coordinate
    return np.interp(times, mt, my, left=np.nan, right=np.nan)


def compute_lag(fixations, mouse_events, offsets_ms=None, step_ms=50):
    """Cross-correlate gaze Y with mouse Y at varying temporal offsets.
    Returns (best_lag_ms, rmse_curve, offsets).
    Positive lag = cursor follows gaze (expected)."""
    if len(fixations) < 3 or len(mouse_events) < 10:
        return None, None, None
    
    if offsets_ms is None:
        offsets_ms = np.arange(-2000, 2001, step_ms)
    
    # Common timebase
    all_ts = [f[0] for f in fixations] + [m[0] for m in mouse_events]
    t_start = min(all_ts)
    t_end = max(all_ts)
    
    times, gaze_y = resample_fixations(fixations, t_start, t_end, step_ms)
    mouse_y = resample_mouse(mouse_events, times)
    
    # Only use timepoints where both gaze and mouse are valid
    valid = np.isfinite(gaze_y) & np.isfinite(mouse_y)
    if valid.sum() < 20:
        return None, None, None
    
    rmses = []
    for offset in offsets_ms:
        shift = int(offset / step_ms)
        if shift >= 0:
            g = gaze_y[shift:]
            m = mouse_y[:len(g)]
        else:
            m = mouse_y[-shift:]
            g = gaze_y[:len(m)]
        
        v = np.isfinite(g) & np.isfinite(m)
        if v.sum() < 10:
            rmses.append(np.nan)
            continue
        rmses.append(np.sqrt(np.mean((g[v] - m[v])**2)))
    
    rmses = np.array(rmses)
    valid_rmses = np.isfinite(rmses)
    if not valid_rmses.any():
        return None, None, None
    
    best_idx = np.nanargmin(rmses)
    best_lag = offsets_ms[best_idx]
    
    return best_lag, rmses, offsets_ms


# Test on one trial
fix = load_fixations('p004-b2-t3')
mouse = load_mouse_positions('p004-b2-t3')
lag, rmses, offsets = compute_lag(fix, mouse)
if lag is not None:
    print(f'p004-b2-t3: best lag = {lag}ms')
    plt.figure(figsize=(8, 3))
    plt.plot(offsets, rmses)
    plt.axvline(lag, color='r', linestyle='--', label=f'min RMSE at {lag}ms')
    plt.xlabel('Offset (ms) — positive = cursor follows gaze')
    plt.ylabel('RMSE (px)')
    plt.title('Gaze-cursor cross-correlation (Y axis)')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Compute lag for all trials ──────────────────────────────────

trial_ids = get_trial_ids()
results = []

for tid in trial_ids:
    try:
        fix = load_fixations(tid)
        mouse = load_mouse_positions(tid)
        lag, rmses, offsets = compute_lag(fix, mouse)
        if lag is not None:
            pid = parse_participant(tid)
            results.append({
                'trial_id': tid,
                'participant': pid,
                'lag_ms': lag,
                'n_fixations': len(fix),
                'n_mouse': len(mouse),
                'min_rmse': float(np.nanmin(rmses)),
            })
    except Exception as e:
        pass

print(f'Computed lag for {len(results)} / {len(trial_ids)} trials')

lags = np.array([r['lag_ms'] for r in results])
print(f'Lag: mean={np.mean(lags):.0f}ms, median={np.median(lags):.0f}ms, SD={np.std(lags):.0f}ms')
print(f'Range: {np.min(lags):.0f} to {np.max(lags):.0f}ms')
print(f'Positive (cursor follows gaze): {(lags > 0).sum()} / {len(lags)} ({(lags > 0).mean()*100:.0f}%)')

In [ ]:
# ── Per-participant averages ────────────────────────────────────

from collections import defaultdict

by_participant = defaultdict(list)
for r in results:
    by_participant[r['participant']].append(r['lag_ms'])

participant_lags = {}
for pid, lags_list in sorted(by_participant.items()):
    participant_lags[pid] = {
        'mean_lag': np.mean(lags_list),
        'median_lag': np.median(lags_list),
        'sd_lag': np.std(lags_list),
        'n_trials': len(lags_list),
    }

mean_lags = np.array([v['mean_lag'] for v in participant_lags.values()])
print(f'Per-participant mean lag:')
print(f'  Mean of means: {np.mean(mean_lags):.0f}ms')
print(f'  SD of means: {np.std(mean_lags):.0f}ms')
print(f'  Range: {np.min(mean_lags):.0f} to {np.max(mean_lags):.0f}ms')
print(f'  (Huang et al.: ~700ms mean, 250-1000ms range)')

# Within-participant consistency (ICC proxy: SD of per-participant lags)
sds = np.array([v['sd_lag'] for v in participant_lags.values() if v['n_trials'] >= 5])
print(f'\nWithin-participant SD (n≥5 trials): mean={np.mean(sds):.0f}ms')
print(f'  (Low = stable trait; high = state-dependent)')

In [ ]:
# ── Visualize ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Distribution of trial-level lags
axes[0].hist([r['lag_ms'] for r in results], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(700, color='red', linestyle='--', alpha=0.7, label='Huang et al. 700ms')
axes[0].set_xlabel('Gaze-cursor lag (ms)')
axes[0].set_ylabel('Trials')
axes[0].set_title('Trial-level lag distribution')
axes[0].legend(fontsize=9)

# 2. Per-participant mean lag (sorted)
sorted_pids = sorted(participant_lags.keys(), key=lambda p: participant_lags[p]['mean_lag'])
means = [participant_lags[p]['mean_lag'] for p in sorted_pids]
sds_plot = [participant_lags[p]['sd_lag'] for p in sorted_pids]
axes[1].barh(range(len(means)), means, xerr=sds_plot, color='steelblue', alpha=0.7, ecolor='gray')
axes[1].axvline(700, color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Mean lag (ms)')
axes[1].set_ylabel('Participant (sorted)')
axes[1].set_title('Per-participant lag')
axes[1].set_yticks([])

# 3. Lag vs number of fixations (proxy for trial complexity)
axes[2].scatter([r['n_fixations'] for r in results], [r['lag_ms'] for r in results],
               alpha=0.2, s=10, color='steelblue')
axes[2].set_xlabel('Fixation count')
axes[2].set_ylabel('Lag (ms)')
axes[2].set_title('Lag vs trial length')

plt.tight_layout()
plt.savefig('plot_gaze_cursor_lag.png', dpi=150, bbox_inches='tight')
plt.show()

## Next: correlate with TTI and regression rate

TODO (integrate with existing notebooks):
- Load TTI-to-first-scroll per participant from `orientation_evaluation.ipynb` output
- Correlate per-participant mean lag with mean TTI (hypothesis: r > 0.5)
- Correlate per-participant mean lag with regression rate (hypothesis: optimizers have different lag)
- Correlate per-participant mean lag with LHIPA (cognitive load)
- Test within-participant stability: ICC across trials